# Check completeness property for IG attributions

In [ ]:
from captum.attr import configure_interpretable_embedding_layer
from captum.attr import remove_interpretable_embedding_layer
from matplotlib import pyplot as plt
import torch
import numpy as np
import random

import config
import loader
import attributor
import evaluator

SCOPE = 'global'
SUB_BITS = config.SUB_BITS
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
METHOD_NAME = 'ig'
N_STEPS = 10

## LAAT

In [ ]:
# Load model
model_name = 'laat'
model, dataloader = loader.load_model_and_data(model_name)
def model_wrapper(*args, **kwargs):
    output, _ = model(*args, **kwargs)
    return torch.sigmoid(output[1])
int_emb = configure_interpretable_embedding_layer(model, 'embedding')
model.train()

In [ ]:
# Create IG attribution method
method = attributor.create_method(model_wrapper, SCOPE, METHOD_NAME)

In [ ]:
diffs = []
sums = []
N = 3
for idx, tup in enumerate(dataloader):
    # Attribute only the N first samples of the subset
    if SUB_BITS[idx] == 0 or np.count_nonzero(SUB_BITS[:idx]) > N-1:
        continue
    print("Attributing sample", idx)
    input_embed, base_embed, afa = evaluator.prepare_input(model_name, tup, int_emb)
    preds_input = model_wrapper(input_embed, afa)[0]
    preds_base = model_wrapper(base_embed, afa)[0]
    for target_idx, pred_input in enumerate(preds_input):
        if pred_input.item() > config.THRESH:
            diffs.append(pred_input.item() - preds_base[target_idx].item())
            print("Attributing target_idx", target_idx)
            a = attributor.attribute(SCOPE,
                                     METHOD_NAME,
                                     method,
                                     input_embed,
                                     base_embed,
                                     afa,
                                     target_idx,
                                     n_steps=N_STEPS,
                                     n_samples=None)
            a = a.sum(dim=2).squeeze(0).cpu().detach().numpy()
            sums.append(sum(a))

In [ ]:
# Plot completeness
plt.scatter(sums, diffs)
plt.show()

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)

## CAML

In [ ]:
# Load model
model_name = 'caml'
model, dataloader = loader.load_model_and_data(model_name)
def model_wrapper(*args, **kwargs):
    output, _, _ = model(*args, **kwargs)
    return torch.sigmoid(output)
int_emb = configure_interpretable_embedding_layer(model, 'embed')
model.train()

In [ ]:
# Create IG attribution methods
method = attributor.create_method(model_wrapper, SCOPE, METHOD_NAME)

In [ ]:
diffs = []
sums = []
N = 3
for idx, tup in enumerate(dataloader):
    # Attribute only the N first samples of the subset
    if SUB_BITS[idx] == 0 or np.count_nonzero(SUB_BITS[:idx]) > N-1:
        continue
    print("Attributing sample", idx)
    input_embed, base_embed, afa = evaluator.prepare_input(model_name, tup, int_emb)
    preds_input = model_wrapper(input_embed, afa)[0]
    preds_base = model_wrapper(base_embed, afa)[0]
    for target_idx, pred_input in enumerate(preds_input):
        if pred_input.item() > config.THRESH:
            diffs.append(pred_input.item() - preds_base[target_idx].item())
            print("Attributing target_idx", target_idx)
            a = attributor.attribute(SCOPE,
                                     METHOD_NAME,
                                     method,
                                     input_embed,
                                     base_embed,
                                     afa,
                                     target_idx,
                                     n_steps=N_STEPS,
                                     n_samples=None)
            a = a.sum(dim=2).squeeze(0).cpu().detach().numpy()
            sums.append(sum(a))

In [ ]:
# Plot completeness
plt.scatter(sums, diffs)
plt.show()

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)